# 连接线下设备

In [1]:
from subprocess import run, PIPE
parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')

id_rsa_online = "/Users/skybrid/.ssh/adc_jym/id_rsa_online"
id_rsa_online2 = "/Users/vip/aic-jymadc-keypair-test/id_rsa_online2"
devices = [
    ("111.62.192.12:5988", id_rsa_online),
    # ("111.62.192.12:5689", id_rsa_online),  # 武峥
    # ("111.62.192.12:5697", id_rsa_online), # 凌豪
    # ("111.62.192.12:5688", id_rsa_online),  # 黄迪
    # ("111.62.192.12:5687", id_rsa_online),  # 豪爽
    # ("111.62.192.12:5686", id_rsa_online),  # 豪爽
    # ("120.232.102.68:19678", id_rsa_online2), # 无尽冬日
]

for ip_port, private_key in devices:
    ip, port = ip_port.split(":")
    local_port = f"1{port[-4:]}"
    connect_cmd = f"""
    ssh -L {local_port}:localhost:5555 root@{ip} -p {port} \
    -i {private_key} -Nf -o ServerAliveInterval=60 -o StrictHostKeyChecking=no \
    -o CheckHostIP=no -o UserKnownHostsFile=/dev/null
    adb connect localhost:{local_port}
    """
    # adb -s localhost:{local_port} root
    # print(connect_cmd.strip())
    res = run(connect_cmd.strip(), **parameter)
    if stdout := res.stdout: print(stdout.strip())
    # if stderr := res.stderr: print(stderr)

connected to localhost:15988


# 开始投屏

In [4]:
from subprocess import Popen
from pathlib import Path

filename = f"{Path().resolve()}/scrcpy投屏.sh"
res = Popen(['/System/Applications/Utilities/Terminal.app/Contents/MacOS/Terminal', filename])

# 连接线上设备

In [ ]:
from subprocess import run, PIPE
parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')

private_key = "/Users/vip/aic-jymadc-keypair-test/id_rsa_online"
devices = [
    # "111.62.202.85:5965", # 预发区服更新
    # "111.6.24.11:5913",   # 线上区服更新1
    "111.6.24.11:5912",   # 线上区服更新2
    # "111.6.24.11:5990",   # 三战: 小米
    # "111.6.24.11:5991",   # 三战: 九游
    # "111.6.24.11:5993",   # 三战: 九游
    # "111.6.24.11:5994",   # 三战: 九游
    # "111.6.24.11:5992",   # 三战: 犀游、灵犀
    # "111.62.202.85:5645"  # 小米
    # "111.6.24.11:5996",   # 预发
]
for ip_port in devices:
    ip, port = ip_port.split(":")
    local_port = f"2{port[-4:]}"
    connect_cmd = f"""
    ssh -L {local_port}:localhost:5555 root@{ip} -p {port} \
    -i {private_key} -Nf -o ServerAliveInterval=60 -o StrictHostKeyChecking=no
    adb connect localhost:{local_port}
    """
    # print(connect_cmd)
    res = run(connect_cmd.strip(), **parameter)
    if stdout := res.stdout: print(stdout.strip())
    # if stderr := res.stderr: print(stderr)

connected to localhost:25912
connected to localhost:25996


# 断开连接

In [11]:
import re
from subprocess import run, PIPE
parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')

exclude = [15690]
res = run("adb devices", **parameter)
devices_list_re = re.compile(r"localhost:(\d+)")
devices_list = res.stdout.strip()
port_list = devices_list_re.findall(devices_list)
port_list = filter(lambda x: int(x) not in exclude, port_list)

for port in port_list:
    cmd = f"adb disconnect localhost:{port}"
    res = run(cmd, **parameter)
    if stdout := res.stdout: print(stdout.strip())
    if stderr := res.stderr: print(stderr)

disconnected localhost:25936
disconnected localhost:25996


# 旋转屏幕

In [185]:
import logging
logger = logging.getLogger("airtest")
logger.setLevel(logging.ERROR)

from airtest.core.api import *

auto_setup(devices=["android://localhost:5037/localhost:15690"])
DEV = device()
orientation_func = lambda: DEV.display_info["orientation"]
orientation = orientation_func()
if orientation in (1, 3):
    DEV.shell("settings put system accelerometer_rotation 0")
    DEV.shell("settings put system user_rotation 0")
    DEV.shell("settings put system accelerometer_rotation 1")
    sleep(0.5)
    print(f"切换横屏 {orientation} --> {orientation_func()}")
else:
    DEV.shell("settings put system accelerometer_rotation 0")
    DEV.shell("settings put system user_rotation 3")
    DEV.shell("settings put system accelerometer_rotation 1")
    sleep(0.5)
    print(f"切换竖屏 {orientation} --> {orientation_func()}")

切换竖屏 0 --> 3


# 截屏

In [45]:
from subprocess import run, Popen, call
import datetime

timeNow = lambda: datetime.datetime.now().strftime('tpl%Y%m%d%H%M%S')
local_port = '15988'
device = "localhost:"+ local_port

download_path = "~/adb_image"

def cmd_func(filename):
    return f"""
    adb -s {device} shell screencap -p /sdcard/{filename}
    adb -s {device} pull /sdcard/{filename} {download_path}
    adb -s {device} shell rm /sdcard/{filename}
    """

filename = f"{timeNow()}.png"
cmd = cmd_func(filename)
run(cmd, shell=True)
print()
# res = Popen(['/System/Applications/Preview.app/Contents/MacOS/Preview', f"~/Downloads/{filename}"])

/sdcard/tpl20250919102814.png: 1 file pulled, 0 skipped. 5.7 MB/s (3430447 bytes in 0.572s)



# 活动

In [41]:
from airtest.core.api import *

import logging
logger = logging.getLogger("airtest")
logger.setLevel(logging.ERROR)

auto_setup(devices=["android://localhost:5037/localhost:15690"])

In [45]:
import re
from pprint import pprint


def activityApps(is_front=True):
    """
    正在活动app
    :return: applist
    """
    try:
        # 前台活动
        if is_front:
            activityComponent = shell("dumpsys window | grep mCurrentFocus")
            applist = re.findall(r"mCurrentFocus.*u0\s(.*?)/", activityComponent)

        # 全部活动
        else:
            activityComponent = shell("dumpsys activity top | grep ACTIVITY")
            applist = re.findall(r"ACTIVITY\s(.*?)/", activityComponent)

    except Exception as e:
        print(f"---activityAppsError: {e}")
        return
    return applist

# shell("pm clear com.xiaomi.gamecenter.sdk.service")
# shell("wm density 300")
shell("wm density 450")
packageName = "com.netease.sky.bilibili"
# stop_app(packageName)
# start_app(packageName)

applist = activityApps(0)
pprint(applist)

['com.estrongs.android.pop',
 'com.estrongs.android.pop',
 'com.mmbox.xbrowser',
 'ginlemon.flowerfree',
 'com.netease.sky']


## 一键清理

In [46]:
exclude = [
    "ginlemon.flowerfree",
]
applist = activityApps(0)
for app in applist:
    if app in exclude:
        continue
    stop_app(app)

# 下载APP

In [27]:
from subprocess import run
import pyperclip

device = "localhost:15620"

def install_app():
    url = pyperclip.paste()
    if not url.startswith("http"):
        print("Invalid URL")
        return

    cmd = [
        f"adb -s {device} shell /data/curl -k -o /sdcard/base.apk {url}",
        # f"adb -s {device} shell pm install /sdcard/base.apk",
        # f"adb -s {device} shell rm /sdcard/base.apk"
    ]
    for c in cmd:
        print(c)
        run(c.strip(), shell=True)

install_app()

adb -s localhost:15620 shell /data/curl -k -o /sdcard/base.apk https://pkg.biligame.com/games/gy_0.14.0_18_20250418_032317_49233.apk


/system/bin/sh: /data/curl: inaccessible or not found


# 安装APP

In [ ]:
from subprocess import run

local_port = [
    "15620",
]

def install_app():
    for port in local_port:
        cmd = f"adb -s localhost:{port} shell pm install /sdcard/base.apk && adb -s localhost:{port} shell rm /sdcard/base.apk"
        run(cmd.strip(), shell=True)
        print(f"localhost:{port} install successfully")

install_app()

avc:  denied  { read } for  scontext=u:r:system_server:s0 tcontext=u:object_r:fuse:s0 tclass=file permissive=1


Success
localhost:15620 install successfully


# 下载图片

In [30]:
import requests
from urllib import parse
from pathlib import Path
import pyperclip
import os

def download_image(url=None):
    # url = pyperclip.paste()
    url = url.split("?")[0]
    url_split = os.path.split(url)
    if not all(url_split):
        print("Invalid URL")
        return
    if not url.startswith("http"):
        print("Invalid URL")
        return

    save_path = Path("/Users/vip/Downloads") / parse.unquote(url_split[1])
    response = requests.get(url)
    save_path.write_bytes(response.content)
    # print("Save Successfully")

urls = ["https://static.jiaoyimao.com/jym-oss/resource/vmos/20250902/adc_8985347/adc_catch_%E8%AE%BE%E6%96%BD_1756823983127043.jpg"]
for url in urls:
    download_image(url)

# 剪贴板

In [88]:
from pathlib import Path
import logging
logger = logging.getLogger("airtest")
logger.setLevel(logging.ERROR)

from airtest.core.api import *

auto_setup(devices=["android://localhost:5037/localhost:15620"])
DEV = device()

install(str(Path().resolve() / "clipper.apk"), install_options=["-r", "-f"])
shell("am startservice ca.zgrs.clipper/.ClipboardService")
shell(f'am broadcast -a clipper.set --es text ""')
clipper = shell('am broadcast -a clipper.get')
print(clipper)

# from airtest.core.api import *
# auto_setup(devices=["android://localhost:5037/localhost:15620"])

Broadcasting: Intent { act=clipper.get flg=0x400000 }
Broadcast completed: result=0, data=""



In [40]:
text_1 = "test_clipboard"
# set_clipboard(text_1) # 设置剪切板内容

get_text = get_clipboard() # 获得剪切板内容
print(get_text)  # -> test_clipboard

# 剪切板粘贴接口
# paste()  # => text(get_clipboard())

[17:51:01][DEBUG]<airtest.core.android.adb> /Applications/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5037 -s localhost:15620 shell app_process -Djava.class.path=/data/app/com.netease.nie.yosemite-5fVOZbjCpdvVnBjrrJkqsQ==/base.apk / com.netease.nie.yosemite.control.Control --DEVICE_OP clipboard_get


Could not invoke method java.lang.reflect.InvocationTargetException
null


In [57]:
a = "adc_catch_%E7%99%BD%E8%9C%A1%E7%83%9B%E6%95%B0%E9%87%8F_1735135920056543"
print(parse.unquote(a))

adc_catch_白蜡烛数量_1735135920056543


# Airtest滑动测试

In [157]:
import os
import logging
logger = logging.getLogger("airtest")
logger.setLevel(logging.ERROR)

from airtest.core.api import *
from airtest.core.android.touch_methods.base_touch import DownEvent, MoveEvent, SleepEvent, UpEvent

auto_setup(devices=["android://localhost:5037/localhost:15690"])
# auto_setup(devices=["android://localhost:5037/localhost:15689"])
# auto_setup(devices=["android://localhost:5037/localhost:25939"])

DEV = device()
display_info = DEV.display_info
width = display_info['width']
height = display_info['height']
screen_vertical = lambda: DEV.display_info["orientation"] in (1, 3)
SCREENSIZE = (height, width) if screen_vertical() else (width, height)
SCREENHALF = SCREENSIZE[0] // 2, SCREENSIZE[1] // 2
SCREENBOX = (0, 0), SCREENSIZE

# Airtest滑动
class SwipeAirtest:
    def __init__(self, start_point, delta_y=0.0, delta_x=0.0, end_point=None, offset=None, duration=None, hold_sec=0.5, sec=0.5):
        self.start_point = start_point
        self.end_point = end_point
        self.delta_y = delta_y
        self.delta_x = delta_x
        self.hold_sec = hold_sec
        self.sec = sec
        self.offset = offset
        self.duration = duration
        self.start_point_ = None
        self.screen_vertical = screen_vertical()

    def processing(self):
        # 竖屏
        if self.screen_vertical:
            self.start_point_ = self.start_point[1], SCREENHALF[0] - self.start_point[0] + SCREENHALF[0]
            if not self.end_point:
                self.end_point = self.start_point_[0] + self.delta_y, self.start_point_[1] - self.delta_x
        # 横屏
        else:
            if not self.end_point and any((self.delta_x, self.delta_y)):
                self.end_point = self.start_point[0] + self.delta_x, self.start_point[1] + self.delta_y

    @staticmethod
    def __swipe_move(from_xy, to_xy, duration, steps=5):
        from_x, from_y = from_xy
        to_x, to_y = to_xy

        ret = []
        interval = duration / (steps + 1)
        delta_x, delta_y = to_x - from_x, to_y - from_y

        for i in range(1, steps):
            ret.append(MoveEvent((from_x + delta_x * i / steps, from_y + delta_y * i / steps)))
            ret.append(SleepEvent(interval))
        ret.extend([MoveEvent((to_x, to_y)), SleepEvent(interval)])
        return ret

    def __swipe_along(self, coordinates_list, duration, steps=5):
        from_xy = coordinates_list[0]
        swipe_events = [DownEvent(from_xy), SleepEvent(0.1)]
        for to_xy in coordinates_list[1:]:
            swipe_events.extend(self.__swipe_move(from_xy, to_xy, duration=duration, steps=steps))
            from_xy = to_xy
        swipe_events.append(SleepEvent(self.hold_sec))
        DEV.touch_proxy.perform(swipe_events)

    # 按下并滑动
    def down_swipe(self):
        self.processing()
        to_x, to_y = self.end_point

        if self.offset and isinstance(self.offset, int):
            offset = self.end_point
            # 横屏向上滑动
            if not self.screen_vertical and self.delta_y < 0: offset = (to_x, to_y + self.offset)
            # 横屏向下滑动
            elif not self.screen_vertical and self.delta_y > 0: offset = (to_x, to_y - self.offset)
            # 横屏向左滑动
            elif not self.screen_vertical and self.delta_x < 0: offset = (to_x + self.offset, to_y)
            # 横屏向右滑动
            elif not self.screen_vertical and self.delta_x > 0: offset = (to_x - self.offset, to_y)

            # 竖屏向上滑动
            elif self.screen_vertical and self.delta_y < 0: offset = (to_x + self.offset, to_y)
            # 竖屏向下滑动
            elif self.screen_vertical and self.delta_y > 0: offset = (to_x - self.offset, to_y)
            # 竖屏向左滑动
            elif self.screen_vertical and self.delta_x < 0: offset = (to_x, to_y - self.offset)
            # 竖屏向右滑动
            elif self.screen_vertical and self.delta_x > 0: offset = (to_x, to_y + self.offset)

            self.__swipe_along([self.start_point_ or self.start_point, offset, (to_x, to_y)], duration=self.duration or 0.5)
        else:
            self.__swipe_along([self.start_point_ or self.start_point, self.end_point], duration=self.duration or 0.5)

    # 放开
    def up(self):
        DEV.touch_proxy.perform([UpEvent(0), SleepEvent(self.sec)])

    def reset(self):
        self.delta_x = 0
        self.delta_y = 0
        self.end_point = None

    def __enter__(self):
        self.down_swipe()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.up()

    def __call__(self, start_point=None, end_point=None, delta_y=0.0, delta_x=0.0, hold_sec=None, sec=None):
        if start_point: self.start_point = start_point
        if end_point: self.end_point = end_point
        if delta_y:
            self.reset()
            self.delta_y = delta_y
        if delta_x:
            self.reset()
            self.delta_x = delta_x
        if hold_sec: self.hold_sec = hold_sec
        if sec: self.sec = sec
        self.down_swipe()
        self.up()

def air_swipe_fast(start_point, delta_y=0.0, delta_x=0.0, end_point=None, offset=None, duration=None, hold_sec=0.5, sec=0.5):
    SwipeAirtest(
        start_point,
        delta_y=delta_y,
        delta_x=delta_x,
        end_point=end_point,
        offset=offset,
        duration=duration,
        hold_sec=hold_sec,
        sec=sec)()

In [ ]:
start_point = (856, 964)

def swipe_reset():
    for _ in range(1):
        air_swipe_fast(start_point, delta_y=10000, hold_sec=0, sec=0.5)

swipe_reset()

In [61]:
swipe_reset()
sleep(2)
sp = SwipeAirtest(start_point, offset=-1)
sp(delta_y=-845)
sp(delta_y=-1125)
sp(delta_y=-1125)
sp(delta_y=-845)
sp(delta_y=-1125)
sp(delta_y=-1125)
sp(delta_y=-1125)
sp(delta_y=-1125)
sp(delta_y=-1125)

# sp(delta_y=-899)
# sp(delta_y=-28)
# sp(delta_y=-899)
# sp(delta_y=-899)
# sp(delta_y=-117)
# sp(delta_y=-899)

In [ ]:
sp(delta_y=-845)

In [86]:
sp(delta_y=-1125)

In [114]:
delta_y = 294
sp(delta_y=-delta_y, sec=0)
# sp(delta_y=delta_y, sec=0)

# iOS滑动测试

In [ ]:
from iOS_swipe_test import *

# ENV.iOSDeviceId = "00008101-00061DCE11C0001E"
ENV.iOSDeviceId = "00008101-000975213AC0001E"

# 横屏
os.environ.update(
    screenRotation="H",
)

app = Application()
app.start()

def air_swipe_fast(start_point, delta_y=0.0, delta_x=0, end_point=None, offset=None, duration=None, hold_sec=0.5, sec=0.5):
    if not end_point and any((delta_x, delta_y)):
        end_point = start_point[0] + delta_x, start_point[1] + delta_y
    ios_swipe(start_point, end_point, duration=duration or 0.5, sec=sec)

[2025-08-06 10:14:28,052] client.py:44 5985:MainThread:140704341965696 [INFO]Init civita sdk V1.2.101
[2025-08-06 10:14:28,054] logger.py:61 5985:MainThread:140704341965696 [INFO]Shrink log files begin
[2025-08-06 10:14:28,056] logger.py:74 5985:MainThread:140704341965696 [INFO]/Users/vip/logs/civita/pysdk/pysdk.2025_08_06.log modified at 2025-08-13 10:14:28.054780, -7 days ago
[2025-08-06 10:14:28,057] logger.py:79 5985:MainThread:140704341965696 [INFO]Shrink log files done


---iOS设备_init_client iOS设备_init_client
---iOS设备_setup_device_agent_start iOS设备_setup_device_agent_start
---iOS设备_setup_cb iOS设备_setup_cb
---连接iOS设备_start 连接iOS设备
---连接iOS设备_session_timeout
---连接iOS设备_session_timeout 连接iOS设备超时


In [2]:
# 强制释放设备
force_release_device()

[2025-08-06 09:53:24,392] client.py:164 4595:MainThread:140704341965696 [INFO]Release device 00008101-000975213AC0001E
[2025-08-06 09:53:24,394] client.py:82 4595:MainThread:140704341965696 [INFO]Request api=device/00008101-000975213AC0001E/release, json=null


---强制释放iOS设备_start 强制释放iOS设备_start


[2025-08-06 09:53:24,578] client.py:88 4595:MainThread:140704341965696 [INFO]Response api=device/00008101-000975213AC0001E/release, code=200, data=b'{"success":true,"code":200,"message":"Success."}'


---强制释放iOS设备 ---
---强制释放iOS设备成功 


In [3]:
for i in range(1, 3):
    air_swipe_fast(((2254, 500)), 2000, sec=0)

In [15]:
y = -430
for i in range(1, 15):
    air_swipe_fast((2254, 900), y, sec=0, duration=1)
    print(i, y)
    # if i == 1 or i % 3 == 0:
    #     y = -256
    # else:
    #     y = -255
    # if 7 <= i <= 9:
    #     y -= 1
    sleep(2)

1 -430
2 -430


KeyboardInterrupt: 

# POCO滑动测试

In [ ]:
import os
import logging
logger = logging.getLogger("airtest")
logger.setLevel(logging.WARN)

from airtest.core.api import *

os_env = os.environ
auto_setup(devices=["android://localhost:5037/localhost:9999"])

DEV = device()
display_info = DEV.display_info
width = display_info['width']
height = display_info['height']
SCREENSIZE = (height, width) if os_env.get("screenRotation") == "vertical" else (width, height)
SCREENHALF = SCREENSIZE[0] // 2, SCREENSIZE[1] // 2
SCREENBOX = (0, 0), SCREENSIZE

from poco.drivers.android.uiautomation import AndroidUiautomationPoco
poco = AndroidUiautomationPoco(screenshot_each_action=False)

from track import MotionTrack

# pos归一化
pos_normalization = lambda pos: (pos[0] / SCREENSIZE[0], pos[1] / SCREENSIZE[1])

# Poco滑动
class SwipePoco:
    def __init__(self, start_point, delta_y=0.0, delta_x=0.0, end_point=None, hold_sec=0.5, sec=0.5):
        self.start_point = start_point
        self.end_point = end_point
        self.delta_y = delta_y
        self.delta_x = delta_x
        self.hold_sec = hold_sec
        self.sec = sec

    def processing(self, direction):
        if not self.end_point and any((self.delta_x, self.delta_y)):
            self.end_point = self.start_point[0] + self.delta_x, self.start_point[1] + self.delta_y

        if direction == "down":
            track = MotionTrack(speed=0.6, is_hold=True)
            track = track.start(pos_normalization(self.start_point))
            track = track.move(pos_normalization(self.end_point))
            track = track.hold(self.hold_sec)
        else:
            track = MotionTrack([pos_normalization(self.end_point)])
            track = track.hold(self.sec)
        poco.apply_motion_tracks([track], accuracy=0.04)

    def reset(self):
        self.delta_x = 0
        self.delta_y = 0
        self.end_point = None

    def down_swipe(self):
        self.processing("down")

    def up(self):
        self.processing("up")

    def __enter__(self):
        self.down_swipe()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.up()

    def __call__(self, start_point=None, end_point=None, delta_y=0.0, delta_x=0.0, hold_sec=None, sec=None):
        if start_point: self.start_point = start_point
        if end_point: self.end_point = end_point
        if delta_y:
            self.reset()
            self.delta_y = delta_y
        if delta_x:
            self.reset()
            self.delta_x = delta_x
        if hold_sec: self.hold_sec = hold_sec
        if sec: self.sec = sec
        self.down_swipe()
        self.up()

def poco_swipe_fast(start_point, delta_y=0.0, delta_x=0.0, end_point=None, sec=1):
    if not end_point and any((delta_x, delta_y)):
        end_point = start_point[0] + delta_x, start_point[1] + delta_y
    poco.swipe(pos_normalization(start_point), pos_normalization(end_point), duration=0.5)
    sleep(sec)

[23:37:08][WARNING]<airtest.core.api> Device:localhost:9999 updated <airtest.core.android.android.Android object at 0x10f8b0490> -> <airtest.core.android.android.Android object at 0x10952d940>


In [ ]:
start_point = (1340, 1140)

SwipeAirtest(start_point, 1000000)()
for i in range(10):
    SwipePoco(
        start_point,
        delta_y=-719,
        sec=2
    )()

# ADB滑动测试

In [ ]:
def adb_swipe(start_point, end_point, sec=1):
    shell(f"input swipe {start_point[0]} {start_point[1]} {end_point[0]} {end_point[1]} {sec * 1000}")

In [22]:
a = """
Physical density: 360
Override density: 450
"""

import re

if dpi := re.search(r"Override density: (\d+)", a):
    print(dpi.group(1))

450


# POCO测试

In [25]:
import logging
logger = logging.getLogger("airtest")
logger.setLevel(logging.ERROR)
from airtest.core.api import *

auto_setup(devices=["android://localhost:5988/localhost:15988"])

from poco.drivers.android.uiautomation import AndroidUiautomationPoco
poco = AndroidUiautomationPoco(use_airtest_input=True, screenshot_each_action=False)

[10:51:14][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 get-state
[10:51:17][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 connect localhost:15988
[10:51:17][INFO]<airtest.core.android.adb> connected to localhost:15988

[10:51:17][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 wait-for-device
[10:51:17][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell getprop ro.build.version.sdk
[10:51:18][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell dumpsys activity top


In [29]:
ball =  poco("android.widget.FrameLayout").offspring("android.widget.Button")
if ball.exists():
    ball.click()

[10:55:16][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell ls -l /data/local/tmp/rotationwatcher.jar
[10:55:16][DEBUG]<airtest.core.android.rotation> install_rotationwatcher skipped
[10:55:16][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell app_process -Djava.class.path=/data/local/tmp/rotationwatcher.jar /data/local/tmp com.example.rotationwatcher.Main
[10:55:16][DEBUG]<airtest.utils.nbsp> [rotation_server]b'270'
[10:55:17][INFO]<airtest.core.android.rotation> update orientation None->3
[10:55:17][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell wm size
[10:55:17][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/a

[11:15:12][DEBUG]<airtest.utils.nbsp> [rotation_server]b'0'
[11:15:12][DEBUG]<airtest.utils.nbsp> [rotation_server]b'270'
[11:15:12][INFO]<airtest.core.android.rotation> update orientation 3->0
[11:15:12][INFO]<airtest.core.android.rotation> update orientation 0->3
[17:02:51][DEBUG]<airtest.utils.nbsp> [rotation_server]b''
[17:02:51][DEBUG]<airtest.utils.nbsp> [airtouch_server]b''
[17:02:52][DEBUG]<airtest.core.android.rotation> orientationWatcher has ended


In [30]:
user_id = poco("android.widget.FrameLayout").offspring("android:id/content").child("android.widget.FrameLayout").child("android.widget.LinearLayout").offspring("android.widget.FrameLayout")[0].child("android.webkit.WebView").child("android.webkit.WebView").child("android.view.View").child("android.widget.TextView")[2]
if user_id.exists():
    a = user_id.get_text()
print(a)

账号: 2239073858


In [33]:
import re
re.sub(r'\D+', '', a)

'2239073858'

[11:12:33][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell am start -n com.netease.open.pocoservice/.TestActivity
[11:12:33][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell am instrument -w -e debug false -e class com.netease.open.pocoservice.InstrumentedTestAsLauncher com.netease.open.pocoservice/androidx.test.runner.AndroidJUnitRunner
[11:15:23][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell am start -n com.netease.open.pocoservice/.TestActivity
[11:15:23][DEBUG]<airtest.core.android.adb> /Users/skybrid/.Trash/AirtestIDE.app/Contents/MacOS/airtest/core/android/static/adb/mac/adb -P 5988 -s localhost:15988 shell am instrument -w -e debug false -e class com.net

In [31]:
pos_back2 = poco("android.widget.FrameLayout").offspring("android:id/content")\
    .child("android.widget.FrameLayout").child("android.widget.LinearLayout")\
    .offspring("android.widget.FrameLayout")[0].child("android.webkit.WebView")\
    .child("android.webkit.WebView").child("android.view.View").child("android.view.View")[0]\
    .child("android.view.View").child("android.view.View")[2]
if pos_back2.exists():
    pos_back2.click()

In [ ]:
node1 = poco("com.huawei.appmarket:id/applistview")
datetime_name = "com.huawei.appmarket:id/hiappbase_subheader_title_left"
title_name = "com.huawei.appmarket:id/ItemTitle"

for node2 in node1.child():
    for name in [datetime_name, title_name]:
        node3 = node2.offspring(name)
        if node3.exists():
            print(node3.get_text())

In [43]:
shouCang_category = {
    "收藏大师": Template(r"tpl1741531656514.png"),
    "收藏家": Template(r"tpl1741531489987.png")
}
shouCang_property = {}.fromkeys(shouCang_category, [])
shouCang_property

{'收藏大师': [], '收藏家': []}

# 屏蔽路径列表

In [18]:
from glob import glob
from pathlib import Path
import os
import json
import pyperclip
import re

def select_numbers(string):
    pieces = re.compile(r"(\d+)").split(string)
    pieces[1::2] = map(int, pieces[1::2])
    return pieces

exclude_list = [
    8788,
    13765,
    15574,
    6586,
    10025,
    2717,
    9207,
]

gameID_list = []
adc_path = Path().resolve().parent.parent
for path in glob(str(adc_path / "*")):
    if Path(path).is_dir():
        gameID = os.path.split(path)[1].split("_")[0]
        if gameID.isdigit():
            gameID_list.append(gameID)
gameID_list = sorted(set(gameID_list), key=select_numbers)

gameID_list = dict(map(lambda x: (f"**/{x}_*", True), filter(lambda x: int(x) not in exclude_list, gameID_list)))
pyperclip.copy(json.dumps(gameID_list))

In [35]:
aaa = "3333"


print(aaa)

NameError: name 'aaa' is not defined

In [49]:
"raise_exception".upper()

'RAISE_EXCEPTION'